## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import models as keras_models, optimizers, callbacks, layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.stats import wilcoxon

## Load Final Dataset

In [ ]:
selected_features = np.load('selected_features_irpso.npy')
df_selected = pd.read_csv('final_selected_data_irpso.csv')

X = df_selected.iloc[:, :-1].values  
y = df_selected['label'].values      

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train_cat = to_categorical(y_train, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)

print(f"Data Ready: X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

In [ ]:
# =====================================================================
# VALIDATION OF SYNTHETIC DATA REALISM VIA KS TESTS (AFTER IRPSO)
# =====================================================================
print("\n" + "="*80)
print("VALIDATING SYNTHETIC DATA REALISM: KS TESTS ON IRPSO-SELECTED SPV FEATURES")
print("="*80)

# Step 1: Reload the full balanced dataset (before train/test split) to access synthetic data
# Recall: Your original real dataset had 47 samples BEFORE any augmentation

n_real_original = 47

# Load the full IRPSO-selected dataset (this includes real + CTGAN + SMOTE)
df_full_balanced = pd.read_csv('final_selected_data_irpso.csv')  # This is your balanced dataset
X_full = df_full_balanced.iloc[:, :-1].values  # All features
y_full = df_full_balanced['label'].values     # All labels

print(f"Full balanced dataset shape (after IRPSO): {X_full.shape}")
print(f"Original real samples: {n_real_original}")

# Step 2: Isolate Real vs. Synthetic Data
X_real = X_full[:n_real_original, :]          # First 47 rows = original real data
X_synthetic = X_full[n_real_original:, :]     # Everything after = CTGAN + SMOTE synthetic data

print(f"Real data shape: {X_real.shape}")
print(f"Synthetic data shape: {X_synthetic.shape}")

# Step 3: Load selected feature indices to map back to original SPV features
selected_indices = np.load('selected_features_irpso.npy')
print(f"IRPSO selected feature indices (from original 86D space): {selected_indices}")

# Step 4: Define which original features are SPV (columns 0-31 in original dataset)
spv_original_indices = np.arange(0, 32) 

# Step 5: Find which of the SELECTED features are SPV features
selected_spv_mask = np.isin(selected_indices, spv_original_indices)
selected_spv_indices_in_reduced_space = np.where(selected_spv_mask)[0]  # Indices in the reduced (IRPSO) space
original_spv_indices_selected = selected_indices[selected_spv_mask]     # Original indices of selected SPV features

if len(selected_spv_indices_in_reduced_space) == 0:
    print("WARNING: No SPV features were selected by IRPSO. Skipping KS tests.")
else:
    print(f"\nSelected SPV features for KS testing:")
    for i, orig_idx in enumerate(original_spv_indices_selected):
        print(f"  Reduced Index: {selected_spv_indices_in_reduced_space[i]}, Original Index: {orig_idx}")

    # Step 6: Perform KS Tests
    from scipy import stats

    ks_results = []

    for idx in selected_spv_indices_in_reduced_space:
        real_data = X_real[:, idx]
        synthetic_data = X_synthetic[:, idx]

        # Perform two-sample KS test
        ks_stat, ks_p = stats.ks_2samp(real_data, synthetic_data)

        # Get original feature index for labeling
        orig_feature_idx = selected_indices[idx]
        feature_name = f"SPV_Channel_{orig_feature_idx}"

        # Determine significance
        significance = "NOT SIGNIFICANT (p > 0.05)" if ks_p > 0.05 else "SIGNIFICANT DIFFERENCE (p ≤ 0.05)"

        print(f"\n{feature_name:<15} | KS Statistic: {ks_stat:.4f} | p-value: {ks_p:.4f} | {significance}")

        # Save for table
        ks_results.append({
            'Feature': feature_name,
            'Original_Index': int(orig_feature_idx),
            'KS_Statistic': ks_stat,
            'p_value': ks_p,
            'Significant_Difference': ks_p <= 0.05
        })

    # Step 7: Save Results to CSV
    import pandas as pd
    df_ks_results = pd.DataFrame(ks_results)
    df_ks_results.to_csv('ks_test_results_IRPSO_selected_SPV.csv', index=False)
    print(f"\n KS test results saved to 'ks_test_results_IRPSO_selected_SPV.csv'")

    # Step 8: Summary
    n_tests = len(ks_results)
    n_not_significant = sum(1 for r in ks_results if r['p_value'] > 0.05)
    print(f"\n SUMMARY: {n_not_significant}/{n_tests} SPV features show NO significant difference (p > 0.05) between real and synthetic distributions.")
    if n_not_significant == n_tests:
        print("  This provides strong statistical evidence that the synthetic data preserves the distribution of key spectral biomarkers.")
    elif n_not_significant > n_tests / 2:
        print("  This suggests reasonable realism, though some features show divergence.")
    else:
        print("  This indicates potential issues with synthetic data realism for key SPV features.")

print("\n" + "="*80)
print("VALIDATION COMPLETE. Proceeding with model training...")
print("="*80)

## Multilayer Perceptron Neural Network

In [ ]:
# Re-split the original training data into a new training set (80%) and a hold-out validation set (20%)
X_train_hold, X_val_hold, y_train_hold, y_val_hold = train_test_split(
    X_train, y_train_cat, 
    test_size=0.20, 
    random_state=42, 
    stratify=np.argmax(y_train_cat, axis=1)
)

# Function to build the model
def build_model(hp):
    model = keras_models.Sequential()
    model.add(layers.Dense(
        hp.Int('units1', min_value=256, max_value=1024, step=128), 
        activation='relu', 
        input_shape=(X_train_hold.shape[1],)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.4, step=0.1)))

    model.add(layers.Dense(
        hp.Int('units2', min_value=128, max_value=512, step=128), 
        activation='relu'
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.4, step=0.1)))

    model.add(layers.Dense(
        hp.Int('units3', min_value=64, max_value=256, step=64), 
        activation='relu'
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(hp.Float('dropout3', min_value=0.1, max_value=0.4, step=0.1)))

    model.add(layers.Dense(2, activation='softmax'))

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=hp.Choice('learning_rate', [0.01, 0.001, 0.0001])
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Set up the tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=50,
    factor=3,
    directory='hyperparameter_tuning',
    project_name='irpso_nn'
)

# Callbacks for training
callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, min_lr=1e-6)
]

# Tune the hyperparameters using the new training data and hold-out validation set
tuner.search(
    X_train_hold, y_train_hold, 
    epochs=50, 
    validation_data=(X_val_hold, y_val_hold), 
    callbacks=callbacks_list
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Run the model multiple times for stability
num_runs = 5
metrics_results = { 
    'accuracy': [], 'sensitivity': [], 'specificity': [], 
    'precision': [], 'f1': [], 'auc_roc': [], 'pr_auc': [], 'mcc': [] 
}

for i in range(num_runs):
    print(f"\nTraining Model - Run {i+1}/{num_runs}")

    # Build a new model instance using the best hyperparameters
    model = tuner.hypermodel.build(best_hps)
    
    # Train the model on the training hold-out set with validation on X_val_hold
    history = model.fit(X_train_hold, y_train_hold, epochs=300, batch_size=32,
                        validation_data=(X_val_hold, y_val_hold), callbacks=callbacks_list, verbose=0)

    
    # Make predictions on the hold-out validation set
    y_val_pred_prob = model.predict(X_val_hold)
    y_val_pred = np.argmax(y_val_pred_prob, axis=1)
    y_val_true = np.argmax(y_val_hold, axis=1)
    
    # Evaluate the model on the validation set
    test_loss, test_acc = model.evaluate(X_val_hold, y_val_hold, verbose=0)
    sensitivity = recall_score(y_val_true, y_val_pred)
    precision = precision_score(y_val_true, y_val_pred)
    f1 = f1_score(y_val_true, y_val_pred)
    mcc = matthews_corrcoef(y_val_true, y_val_pred)
    auc_roc = roc_auc_score(y_val_true, y_val_pred_prob[:, 1])
    pr_auc = average_precision_score(y_val_true, y_val_pred_prob[:, 1])
    
    # Compute specificity from the confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_val_true, y_val_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0

    # Store results for this run
    metrics_results['accuracy'].append(test_acc)
    metrics_results['sensitivity'].append(sensitivity)
    metrics_results['specificity'].append(specificity)
    metrics_results['precision'].append(precision)
    metrics_results['f1'].append(f1)
    metrics_results['auc_roc'].append(auc_roc)
    metrics_results['pr_auc'].append(pr_auc)
    metrics_results['mcc'].append(mcc)
    
    print(f"Run {i+1} metrics:")
    print(f" - Accuracy: {test_acc:.4f}")
    print(f" - Sensitivity (Recall): {sensitivity:.4f}")
    print(f" - Specificity: {specificity:.4f}")
    print(f" - Precision: {precision:.4f}")
    print(f" - F1 Score: {f1:.4f}")
    print(f" - AUC-ROC: {auc_roc:.4f}")
    print(f" - PR-AUC: {pr_auc:.4f}")
    print(f" - MCC: {mcc:.4f}")

# Report Mean & Standard Deviation of metrics across runs
print("\nFinal Performance Metrics (Mean ± Std across 5 runs):")
for metric_name, values in metrics_results.items():
    mean_score = np.mean(values)
    std_score = np.std(values)
    print(f"{metric_name.capitalize()}: {mean_score:.4f} ± {std_score:.4f}")

In [ ]:
baseline = 0.5

print("\nWilcoxon Signed-Rank Test Results (vs baseline = 0.50):")
print("-" * 60)
for metric_name, values in metrics_results.items():
    try:
        stat, p_value = wilcoxon(values, alternative='greater', zero_method='wilcox')
        mean_val = np.mean(values)
        print(f"{metric_name.capitalize():<12}: stat = {stat:.4f}, p = {p_value:.4f} | mean = {mean_val:.4f}")
        if p_value < 0.05:
            print(f" => Statistically significant: {metric_name} > {baseline}\n")
        else:
            print(f" => Not statistically significant: cannot conclude {metric_name} > {baseline}\n")
    except ValueError as e:
        print(f"{metric_name.capitalize():<12}: Wilcoxon test failed — {str(e)}\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.boxplot(data=metrics_results['accuracy'], color='skyblue', width=0.3)
sns.stripplot(data=metrics_results['accuracy'], color='black', jitter=True, size=6)

plt.axhline(0.5, color='red', linestyle='--', label='Baseline (0.5)')
plt.title('Model Accuracy Across Runs vs. Baseline', fontsize=14)
plt.ylabel('Accuracy')
plt.xticks([0], ['Accuracy'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
print(f"\n{' MLP Model Summary ':=^50}")
print(f"\nArchitecture:")
print(f"Input: {X_train.shape[1]} features")
print(f"Hidden 1: {best_hps.get('units1')} units (Dropout: {best_hps.get('dropout1'):.1f})")
print(f"Hidden 2: {best_hps.get('units2')} units (Dropout: {best_hps.get('dropout2'):.1f})")
print(f"Hidden 3: {best_hps.get('units3')} units (Dropout: {best_hps.get('dropout3'):.1f})")
print(f"Output: 2 units (softmax)")
print(f"\nTraining:")
print(f"Learning Rate: {best_hps.get('learning_rate'):.0e}")
print(f"Epochs Trained: {len(history.history['val_accuracy'])}")

plt.figure(figsize=(8, 4))
layers = ['Input'] + [f'Hidden {i+1}' for i in range(3)] + ['Output']
units = [X_train.shape[1]] + [best_hps.get(f'units{i+1}') for i in range(3)] + [2]

plt.bar(layers, units, color=['#1f77b4', '#ff7f0e', '#ff7f0e', '#ff7f0e', '#2ca02c'])
plt.title('Model Architecture', fontweight='bold')
plt.ylabel('Units')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
sns.set_style("whitegrid")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color="#3498DB", marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color="#E74C3C", marker='s')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2, color="#2ECC71", marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color="#F39C12", marker='s')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_val_true, y_val_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_val_true, y_val_pred_prob[:, 1])

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#3498DB', linewidth=2, label=f'AUC = {auc_roc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, _ = precision_recall_curve(y_val_true, y_val_pred_prob[:, 1])

plt.figure(figsize=(6, 5))
plt.plot(recall_vals, precision_vals, color='#E74C3C', linewidth=2, label=f'PR AUC = {pr_auc:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
lrs = [0.01 * (0.1 ** (i // 2)) for i in range(len(history.history['loss']))]

plt.figure(figsize=(6, 5))
plt.plot(lrs, color='#2ECC71', linewidth=2, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
plt.figure(figsize=(20, 14))
df_selected.hist(bins=30, figsize=(20, 14), color="#3498DB", edgecolor="black")
plt.suptitle("Feature Distributions", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
sns.pairplot(df_selected, diag_kind='kde', corner=True)
plt.suptitle("Pairwise Relationships Between Features", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_selected.corr(), annot=False, cmap="Reds", fmt='.2f', vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=df_selected, palette="Set2")
plt.title("Target Variable Distribution", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
explainer = shap.Explainer(model, X_train)  
shap_values = explainer(X_test)  

In [ ]:
feature_names = df_selected.columns[:X_test.shape[1]]
shap_values_class_1 = shap_values.values[..., 1]  
shap.summary_plot(shap_values_class_1, X_test, feature_names=feature_names)

In [ ]:
shap_importance = np.abs(shap_values.values).mean(axis=(0, 2))
shap_sorted_idx = np.argsort(shap_importance)[::-1]
shap_cumulative = np.cumsum(shap_importance[shap_sorted_idx]) / np.sum(shap_importance)

plt.figure(figsize=(10, 6))
plt.bar(range(len(selected_features)), shap_importance[shap_sorted_idx], color='b', alpha=0.7, label='SHAP Importance')
plt.plot(range(len(selected_features)), shap_cumulative, color='r', marker='o', label='Cumulative Importance')
plt.axhline(0.8, color='g', linestyle='--', label='80% Threshold')
plt.xlabel("Feature Index", fontsize=12)
plt.ylabel("SHAP Value Importance", fontsize=12)
plt.title("Pareto Plot of Feature Importance", fontsize=14, fontweight='bold')
plt.legend()
plt.show()

## Traditional Machine Learning Models

In [ ]:
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
)
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, make_scorer
)

# Define specificity scorer
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    # Handle cases where confusion matrix might not be 2x2 (e.g., only one class predicted)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        # Return 0 or NaN or handle appropriately if specificity cannot be calculated
        return 0.0

specificity_scorer = make_scorer(specificity_score)

# Model definitions and hyperparameter grids
def get_param_grids():
   # Make sure this includes "XGBoost" as a key
   return {
        "RandomForest": (
            RandomForestClassifier(random_state=42, n_jobs=-1),
            {"n_estimators": [200, 300, 500],
             "max_depth": [5, 10, 15, None]}
        ),
       "LogisticRegression": (LogisticRegression(random_state=42, solver='liblinear', max_iter=1000),
            {"C": [0.001, 0.01, 0.1, 1, 10, 100],
             "penalty": ["l1", "l2"]}  
        ),
         "LinearSVC": (LinearSVC(penalty='l2', loss='squared_hinge', dual=False,
                      max_iter=10000, random_state=42),
            {"C": [0.001, 0.01, 0.1, 1, 10, 100]}
        ),
        "XGBoost": (
            xgb.XGBClassifier(eval_metric='mlogloss', tree_method='hist',
                              random_state=42, n_jobs=-1),
            {"n_estimators": [200, 300, 500],
             "learning_rate": [0.01, 0.1, 0.3],
             "max_depth": [6, 9, 12]}
        ),
        "AdaBoost": (
            AdaBoostClassifier(random_state=42),
            {"n_estimators": [200, 300, 500],
             "learning_rate": [0.01, 0.1, 0.3]}
        )
    }


# Cross-validation settings
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision', 
    'mcc': make_scorer(matthews_corrcoef),
    'specificity': specificity_scorer
}

# Storage for results
results = {}
scores_per_model = {}
# ADD DICTIONARY TO STORE BEST MODELS 
best_models_dict = {}

# Let's assume y_train is the correct single-label format here.
if 'y_train_ml' in locals() or 'y_train_ml' in globals():
     y_train_for_fit = y_train_ml
elif 'y_train' in locals() or 'y_train' in globals():
     # Check if y_train is already single-label
     if len(y_train.shape) == 1 or y_train.shape[1] == 1:
          y_train_for_fit = y_train
     else: # Assume it's one-hot encoded like y_train_cat
          y_train_for_fit = np.argmax(y_train, axis=1)
else:
     raise NameError("Training labels (y_train or y_train_ml) not found.")


# Tuning and evaluation
for name, (model, param_grid) in get_param_grids().items():
    print(f"Tuning {name}...")
    gs = None # Initialize gs variable
    if param_grid:
        gs = GridSearchCV(model, param_grid, cv=rskf,
                          scoring='accuracy', n_jobs=-1, refit=True) # refit=True is default
        gs.fit(X_train, y_train_for_fit) # Use single-label y_train
        best_model = gs.best_estimator_
        print(f"Best params for {name}: {gs.best_params_}\n")
    else:
        # Fit the model if no grid search is specified
        best_model = model.fit(X_train, y_train_for_fit)
        print(f"No hyperparameters to tune for {name}. Using default and fitting.\n")

    # *** STORE THE FITTED BEST ESTIMATOR ***
    best_models_dict[name] = best_model

    # Cross-validation for metrics
    cv_res = cross_validate(
        best_model, X_train, y_train_for_fit, cv=rskf,
        scoring=scoring, return_train_score=False, n_jobs=-1
    )

    scores_per_model[name] = {metric: cv_res[f'test_{metric}'] for metric in scoring}

    # Aggregate results
    metrics = {m: (np.mean(cv_res[f'test_{m}']), np.std(cv_res[f'test_{m}']))
               for m in scoring}
    results[name] = metrics # Store the (mean, std) scores

    # Display metrics
    print(f"{name} Evaluation:")
    for metric, (mean, std) in metrics.items():
        print(f" - {metric}: {mean:.4f} ± {std:.4f}")
    print("\n" + "="*60 + "\n")

# --- (Keep Wilcoxon test code the same, maybe add checks for constant data) ---
print("Wilcoxon Signed-Rank Test Results (vs baseline = 0.50):")
print("-" * 60)
for model_name, model_scores in scores_per_model.items():
    print(f"{model_name}:")
    for metric, scores in model_scores.items():
         # Add check for constant data which causes Wilcoxon error
         unique_diffs = np.unique(scores - 0.5)
         if len(unique_diffs) > 1 or (len(unique_diffs) == 1 and unique_diffs[0] != 0):
              try:
                   stat, p = wilcoxon(scores - 0.5, alternative='greater', zero_method='wilcox') # Added zero_method
                   print(f" - {metric:12s}: stat = {stat:.4f}, p = {p:.4f}")
              except ValueError as e:
                   print(f" - {metric:12s}: Wilcoxon test error: {e}")
         else:
              print(f" - {metric:12s}: Cannot perform Wilcoxon test (constant difference or insufficient variation).")
    print("-" * 60)

In [ ]:
from rich.console import Console
from rich.table import Table
from rich.style import Style

results["MLP"] = {
    "accuracy": np.mean(metrics_results['accuracy']),
    "recall": np.mean(metrics_results['sensitivity']),
    "specificity": np.mean(metrics_results['specificity']),
    "precision": np.mean(metrics_results['precision']),
    "f1": np.mean(metrics_results['f1']),
    "roc_auc": np.mean(metrics_results['auc_roc']),
    "pr_auc": np.mean(metrics_results['pr_auc']),
    "mcc": np.mean(metrics_results['mcc'])
}


# --- Define the mapping from metric_order names to the keys used in the 'results' dictionary ---
metric_key_map = {
    'Accuracy': 'accuracy',
    'Sensitivity (Recall)': 'recall',
    'Specificity': 'specificity',
    'Precision': 'precision',
    'F1 Score': 'f1',
    'MCC': 'mcc',
    'AUC-ROC': 'roc_auc',
    'PR-AUC': 'pr_auc' 
}


# Define the desired output order and display names
metric_order = ['Accuracy', 'Sensitivity (Recall)', 'Specificity', 'Precision',
                'F1 Score', 'MCC', 'AUC-ROC', 'PR-AUC']

# --- Revised DataFrame Creation Logic ---
data_for_df = {}
# Iterate through each model and its associated metrics dictionary in the main 'results'
for model_name, model_results_dict in results.items():
    extracted_metrics = {}
    # For each metric NAME we want in the final table (using metric_order)
    for metric_display_name in metric_order:
        # Find the corresponding KEY used in the 'results' dictionary
        results_key = metric_key_map.get(metric_display_name)

        if results_key:
            # Safely get the value for the metric using the correct key
            metric_value = model_results_dict.get(results_key)

            # Check if the metric value was found
            if metric_value is not None:
                # If it's a tuple (expected for traditional models), extract the mean
                if isinstance(metric_value, tuple) and len(metric_value) >= 1:
                    extracted_metrics[metric_display_name] = metric_value[0]
                # If it's a number (expected for MLP), use it directly
                elif isinstance(metric_value, (float, int, np.number)):
                     extracted_metrics[metric_display_name] = metric_value
                # If the format is unexpected, store NaN
                else:
                    extracted_metrics[metric_display_name] = np.nan
                    # Optional: print a warning for unexpected format
                    # print(f"Warning: Unexpected format for metric '{results_key}' in model '{model_name}'. Value: {metric_value}")
            # If the metric value was not found in the model's results, store NaN
            else:
                extracted_metrics[metric_display_name] = np.nan
                # Optional: print a warning for missing key
                # print(f"Warning: Metric key '{results_key}' not found for model '{model_name}'.")
        else:
             # Metric display name not found in map (indicates error in metric_key_map)
             extracted_metrics[metric_display_name] = np.nan
             print(f"Error: Metric display name '{metric_display_name}' not found in metric_key_map.")

    # Store the extracted metrics (using display names as keys) for the current model
    data_for_df[model_name] = extracted_metrics

# Create the DataFrame from the processed data
results_df = pd.DataFrame.from_dict(data_for_df, orient='index')

# Ensure the columns are in the desired order specified by metric_order
results_df = results_df[metric_order]
# --- End of Revised Logic ---


# =====================
# PLOT WITH CORRECTED DATA
# =====================
plt.figure(figsize=(14, 8)) # Create a figure before plotting
ax = results_df.T.plot(kind='bar', figsize=(14, 8),
                      colormap='viridis',
                      edgecolor='black',
                      alpha=0.85,
                      width=0.8,
                      ax=plt.gca()) # Plot on the current axes

plt.title("Model Performance Comparison", fontsize=16, pad=20)
plt.ylabel("Score Value", fontsize=12)
plt.xlabel("Metrics", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11) # Use ha='right' for better alignment
plt.yticks(fontsize=10)
plt.legend(title="Models", bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout to prevent legend overlap
plt.show()


# =====================
# RICH TABLE WITH CORRECTED DATA AND COLORS
# =====================
console = Console()

# Corrected color scheme - Ensure keys match results_df.index EXACTLY
model_colors = {
    "RandomForest": Style(color="blue"),
    "KNN": Style(color="green"),
    "SVM": Style(color="red"),
    "XGBoost": Style(color="magenta"),
    "AdaBoost": Style(color="cyan"),
    "MLP": Style(color="yellow")
}


table = Table(title="Model Performance Comparison", show_lines=True,
             header_style="bold black")

# Add metric column first
table.add_column("Metric", justify="left", style="bold")

# Add model columns with original colors
# Use results_df.index which contains the correct model names
for model in results_df.index:
    table.add_column(model, justify="center", style=model_colors.get(model, "")) # Use .get for safety

# Add rows in correct order
for metric in metric_order:
    row = [metric]
    for model in results_df.index:
        value = results_df.loc[model, metric]
        # Check if value is NaN before formatting
        if pd.isna(value):
            row.append("[grey]N/A[/grey]") # Style N/A if desired
        else:
            row.append(f"{value:.4f}")
    table.add_row(*row)

console.print(table)

In [ ]:
# Cell for SHAP Analysis using shap.Explainer for Traditional Models
import warnings

# Suppress specific warnings if desired
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')
warnings.filterwarnings("ignore", message="X does not have valid feature names, but RandomForestClassifier was fitted with feature names")

print("--- Starting SHAP Analysis using shap.Explainer ---")

# --- Ensure necessary variables exist ---
if 'best_models_dict' not in locals() or 'best_models_dict' not in globals():
    raise NameError("'best_models_dict' not found. Please run the traditional ML model evaluation cell first.")
if 'X_train' not in locals() or 'X_train' not in globals():
     raise NameError("'X_train' not found. Please run the data preparation cell.")
if 'X_test' not in locals() or 'X_test' not in globals():
     raise NameError("'X_test' not found. Please run the data preparation cell.")

# --- Define feature_names ---
if 'df_selected' in locals() or 'df_selected' in globals():
    n_features = X_test.shape[1]
    if df_selected.shape[1] == n_features + 1:
        feature_names = df_selected.columns[:-1].tolist()
    elif df_selected.shape[1] == n_features:
        feature_names = df_selected.columns.tolist()
    else:
        print(f"Warning: Shape mismatch. Using generic feature names.")
        feature_names = [f'Feature {i}' for i in range(n_features)]
else:
    print("Warning: df_selected DataFrame not found. Using generic feature names.")
    feature_names = [f'Feature {i}' for i in range(X_test.shape[1])]
print(f"Using {len(feature_names)} feature names for plots.")

# --- List of models to explain ---
models_to_explain = ["RandomForest", "LogisticRegression", "LinearSVC", "XGBoost", "AdaBoost"]

# --- Background data for explainers ---
# Use a sample of X_train as background data
background_data_shap = shap.sample(X_train, 100)
print(f"Using background data shape for explainers: {background_data_shap.shape}")

# --- Loop through models and run SHAP analysis ---
for model_name in models_to_explain:
    print(f"\n" + "="*30)
    print(f"--- Running SHAP Analysis for {model_name} ---")
    print("="*30)

    if model_name not in best_models_dict:
        print(f"Model {model_name} not found in best_models_dict. Skipping.")
        continue

    model = best_models_dict[model_name]
    explainer = None
    shap_values = None # This will hold the Explanation object or NumPy array
    shap_values_for_plot = None # This will hold the specific values for plotting (e.g., class 1)
    calculation_successful = False

    try:
        # --- Initialize shap.Explainer ---
        print("Initializing shap.Explainer...")
        # Provide background data as masker for better expected value calculation
        explainer = shap.Explainer(model, background_data_shap)
        print("Calculating SHAP values...")
        shap_values = explainer(X_test) # Calculate Explanation object
        print("SHAP values calculated successfully with shap.Explainer.")
        calculation_successful = True

    except Exception as e:
        print(f"shap.Explainer failed for {model_name}: {e}")
        # --- Fallback for Tree Models if AdditivityError occurred ---
        if "Additivity check failed" in str(e) and isinstance(model, (RandomForestClassifier, xgb.XGBClassifier, AdaBoostClassifier)):
            print(f"AdditivityError detected for {model_name}. Retrying with shap.TreeExplainer(check_additivity=False)...")
            try:
                tree_explainer = shap.TreeExplainer(model)
                # shap_values will now be a NumPy array or list of arrays
                shap_values = tree_explainer.shap_values(X_test, check_additivity=False)
                print("SHAP values calculated successfully with TreeExplainer fallback.")
                calculation_successful = True
            except Exception as e2:
                print(f"TreeExplainer fallback also failed for {model_name}: {e2}")
        elif isinstance(model, AdaBoostClassifier):
             # Fallback to KernelExplainer for AdaBoost if TreeExplainer is not supported
             print(f"TreeExplainer likely not supported for {model_name}. Trying shap.KernelExplainer (may be slow)...")
             try:
                 if hasattr(model, "predict_proba"):
                     def predict_proba_wrapper(X): return model.predict_proba(X)
                     kernel_explainer = shap.KernelExplainer(predict_proba_wrapper, background_data_shap)
                     shap_values = kernel_explainer.shap_values(X_test, nsamples=100) # Adjust nsamples as needed
                     print("SHAP values calculated successfully with KernelExplainer fallback.")
                     calculation_successful = True
                 else:
                     print(f"Model {model_name} lacks predict_proba required by KernelExplainer. Skipping.")
             except Exception as e3:
                 print(f"KernelExplainer fallback failed for {model_name}: {e3}")


    # --- Process SHAP values and Plot ---
    if calculation_successful and shap_values is not None:
        try:
            # Determine structure of shap_values (Explanation object or array/list)
            if hasattr(shap_values, 'values'): # It's an Explanation object
                shap_values_arr = shap_values.values
            elif isinstance(shap_values, list): # It's a list (likely from Tree/Kernel Explainer for classification)
                shap_values_arr = shap_values[1] # Assume we want class 1
            elif isinstance(shap_values, np.ndarray): # It's a NumPy array (Linear or fallback)
                 shap_values_arr = shap_values
            else:
                 raise TypeError("Unexpected SHAP values format.")

            # Handle shape for plotting (use class 1 if 3D)
            if len(shap_values_arr.shape) == 3 and shap_values_arr.shape[2] == 2:
                shap_values_for_plot = shap_values_arr[..., 1]
                plot_title_suffix = " - Class 1"
            elif len(shap_values_arr.shape) == 2:
                shap_values_for_plot = shap_values_arr
                plot_title_suffix = "" # Or specify if it represents a specific class output
            else:
                 raise ValueError("Unexpected SHAP values array shape.")

            # --- SHAP Summary Plot (DOT/BEESWARM STYLE) ---
            print(f"Generating SHAP Dot Summary Plot ({model_name})...")
            plt.figure()
            shap.summary_plot(shap_values_for_plot, X_test, feature_names=feature_names, show=False)
            plt.title(f"SHAP Summary Plot ({model_name}{plot_title_suffix})")
            plt.show()

            # --- Pareto Plot ---
            print(f"Generating Pareto Plot ({model_name})...")
            shap_importance = np.abs(shap_values_for_plot).mean(axis=0)
            shap_sorted_idx = np.argsort(shap_importance)[::-1]
            shap_cumulative = np.cumsum(shap_importance[shap_sorted_idx]) / np.sum(shap_importance)

            plt.figure(figsize=(10, 6))
            plt.bar(range(len(feature_names)), shap_importance[shap_sorted_idx], alpha=0.7, label='SHAP Importance')
            plt.plot(range(len(feature_names)), shap_cumulative, color='r', marker='o', label='Cumulative Importance')
            plt.axhline(0.8, color='g', linestyle='--', label='80% Threshold')
            if len(feature_names) <= 20: plt.xticks(range(len(feature_names)), np.array(feature_names)[shap_sorted_idx], rotation=90)
            else: plt.xlabel("Feature Rank", fontsize=12)
            plt.ylabel(f"Mean Absolute SHAP Value{plot_title_suffix}", fontsize=12)
            plt.title(f"Pareto Plot of Feature Importance ({model_name})", fontsize=14, fontweight='bold')
            plt.legend()
            plt.tight_layout()
            plt.show()

        except Exception as plot_e:
            print(f"An error occurred during SHAP value processing or plotting for {model_name}: {plot_e}")

    else:
        print(f"SHAP values calculation failed for {model_name}, skipping plots.")


print("\n--- Completed SHAP Analysis ---")

In [ ]:
# Cell to Calculate SHAP Importance, Rank Features, and Plot Rankings

# Suppress specific warnings if desired
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')
warnings.filterwarnings("ignore", message="X does not have valid feature names...")

print("--- Calculating Feature Importance Rankings using SHAP ---")

# --- Ensure necessary variables exist ---
if 'best_models_dict' not in locals() or 'best_models_dict' not in globals():
    raise NameError("'best_models_dict' not found. Run the traditional ML cell first.")
mlp_model_to_explain = None
if 'best_model_mlp' in locals() or 'best_model_mlp' in globals():
     mlp_model_to_explain = best_model_mlp
     print("Using 'best_model_mlp' for MLP analysis.")
elif 'model' in locals() or 'model' in globals() and isinstance(model, tf.keras.Model):
     mlp_model_to_explain = model
     print("Using 'model' (likely last run) for MLP analysis.")
else:
     print("Warning: Trained MLP model not found. Skipping MLP.")

if 'X_train' not in locals() or 'X_train' not in globals(): raise NameError("'X_train' not found.")
if 'X_test' not in locals() or 'X_test' not in globals(): raise NameError("'X_test' not found.")

# --- Define feature_names ---
if 'feature_names' not in locals() or 'feature_names' not in globals():
    if 'df_selected' in locals() or 'df_selected' in globals():
        n_features = X_test.shape[1]
        if df_selected.shape[1] == n_features + 1: feature_names = df_selected.columns[:-1].tolist()
        elif df_selected.shape[1] == n_features: feature_names = df_selected.columns.tolist()
        else: feature_names = [f'Feature {i}' for i in range(n_features)]
    else: feature_names = [f'Feature {i}' for i in range(X_test.shape[1])]
    print(f"Using {len(feature_names)} feature names.")
else:
    print("Using pre-defined 'feature_names'.")

# --- Background data for explainers ---
background_data_shap = shap.sample(X_train, 100)
print(f"Using background data shape for some explainers: {background_data_shap.shape}")

# --- Dictionary to store results ---
feature_rankings = {}

# --- Models to analyze ---
models_to_analyze = ["RandomForest", "LogisticRegression", "LinearSVC", "XGBoost", "AdaBoost"]
if mlp_model_to_explain is not None:
    models_to_analyze.append("MLP")

# --- Loop through models, calculate SHAP, rank features ---
for model_name in models_to_analyze:
    print(f"\n--- Analyzing {model_name} ---")

    model = None
    explainer = None
    shap_values = None # Can be Explanation object or array/list
    shap_values_for_calc = None # Array used for mean(|SHAP|) calculation
    calculation_successful = False

    try:
        # Retrieve model object
        if model_name == "MLP":
             model = mlp_model_to_explain
        elif model_name in best_models_dict:
             model = best_models_dict[model_name]
        else:
             print(f"Model object for {model_name} not found. Skipping.")
             continue

        # Select Explainer and Calculate SHAP values
        # (Using the robust logic with fallbacks)
        if isinstance(model, (RandomForestClassifier, xgb.XGBClassifier)):
            print("Using shap.TreeExplainer...")
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_test, check_additivity=False)
            calculation_successful = True

        elif isinstance(model, (LogisticRegression, LinearSVC)):
             print("Using shap.LinearExplainer...")
             explainer = shap.LinearExplainer(model, background_data_shap)
             shap_values = explainer.shap_values(X_test)
             calculation_successful = True

        elif isinstance(model, AdaBoostClassifier):
             print("Using shap.KernelExplainer (AdaBoost)...")
             if hasattr(model, "predict_proba"):
                 def predict_proba_wrapper(X): return model.predict_proba(X)
                 explainer = shap.KernelExplainer(predict_proba_wrapper, background_data_shap)
                 shap_values = explainer.shap_values(X_test, nsamples=100)
                 calculation_successful = True
             else: print(f"AdaBoost model lacks predict_proba. Skipping.")

        elif isinstance(model, tf.keras.Model) and model_name == "MLP":
             print("Using shap.DeepExplainer...")
             explainer = shap.DeepExplainer(model, background_data_shap)
             shap_values = explainer.shap_values(X_test)
             calculation_successful = True
        else:
             print(f"Model type {type(model)} not explicitly handled. Skipping.")


        # Calculate Importance and Rank if successful
        if calculation_successful and shap_values is not None:
            # *** CORRECTED SHAP VALUE PROCESSING ***
            shap_values_arr = None
            # Check if output is a list (Tree/Kernel/Deep for classification)
            if isinstance(shap_values, list) and len(shap_values) == 2:
                shap_values_arr = shap_values[1] # Use class 1 values
            # Check if output is a NumPy array
            elif isinstance(shap_values, np.ndarray):
                # Check if it's 3D (DeepExplainer sometimes returns this)
                if len(shap_values.shape) == 3 and shap_values.shape[-1] == 2:
                     shap_values_arr = shap_values[..., 1] # Extract class 1 slice
                # Check if it's already 2D (Linear or others)
                elif len(shap_values.shape) == 2:
                     shap_values_arr = shap_values # Use directly
            # Handle Explanation object from unified Explainer (if used in future)
            elif hasattr(shap_values, 'values'):
                 if len(shap_values.values.shape) == 3 and shap_values.values.shape[-1] == 2:
                      shap_values_arr = shap_values.values[..., 1]
                 elif len(shap_values.values.shape) == 2:
                      shap_values_arr = shap_values.values
            #-----------------------------------------

            if shap_values_arr is not None and len(shap_values_arr.shape) == 2:
                 # Calculate mean absolute SHAP value per feature
                 shap_importance = np.abs(shap_values_arr).mean(axis=0)

                 if len(shap_importance) == len(feature_names):
                     # Create DataFrame
                     importance_df = pd.DataFrame({
                         'Feature': feature_names,
                         'Mean_Abs_SHAP': shap_importance
                     })
                     # Sort by importance
                     importance_df_sorted = importance_df.sort_values('Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
                     # Store the ranked DataFrame
                     feature_rankings[model_name] = importance_df_sorted
                     print(f"Feature importance ranking calculated for {model_name}.")
                 else: print(f"Error: Mismatch between SHAP importance ({len(shap_importance)}) and feature names ({len(feature_names)}).")
            else: print(f"Error: Could not extract valid 2D SHAP values array for {model_name}.")
        else:
            print(f"SHAP values calculation failed or yielded None for {model_name}.")

    except Exception as e:
        print(f"An error occurred during SHAP analysis setup or calculation for {model_name}: {e}")


# --- Plotting Feature Importance Rankings ---
print("\n\n" + "="*40)
print("--- Feature Importance Rankings Plot (Top 15) ---")
print("="*40)

if not feature_rankings:
    print("\nNo feature rankings were successfully calculated to plot.")
else:
    num_models = len(feature_rankings)
    # Adjust layout dynamically - aiming for 2 columns
    ncols = 2
    nrows = (num_models + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10 * ncols, 5 * nrows), squeeze=False)
    axes = axes.flatten()
    model_idx = 0

    for model_name, ranking_df in feature_rankings.items():
        ax = axes[model_idx]
        top_n_display = min(15, len(ranking_df))
        subset_df = ranking_df.head(top_n_display)

        # Create horizontal bar plot
        ax.barh(subset_df['Feature'], subset_df['Mean_Abs_SHAP'],
                color=plt.cm.viridis(np.linspace(0, 1, top_n_display)), # Example colormap
                edgecolor='black')
        ax.invert_yaxis() # Highest importance at the top
        ax.set_xlabel("Mean Absolute SHAP Value")
        ax.set_title(f"{model_name} Feature Importance")
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        model_idx += 1

    # Hide any unused subplots if the grid is not full
    for i in range(model_idx, len(axes)):
        fig.delaxes(axes[i])

    plt.tight_layout(pad=3.0) # Add padding between subplots
    plt.show()

print("\n--- Ranking Calculation and Plotting Complete ---")